# ClinicalTrialEnv - PyTorch & RL Integration Demo

This notebook demonstrates how an AI Researcher would instantiate the `ClinicalTrialEnv` via the HuggingFace API, and wrap it into a continuous training loop (similar to OpenAI Gym) where an RL Agent continuously optimizes for statistical efficacy while managing a dynamic patient inclusion budget.

In [1]:
import requests
import time

# Point this to your HuggingFace deployment
BASE_URL = 'https://manasdutta04-clinicaltrialenv.hf.space'

def wrap_env_reset(task_id='task_1'):
    res = requests.post(f"{BASE_URL}/reset", json={"task_id": task_id})
    return res.json()['observation']

def wrap_env_step(action_dict):
    res = requests.post(f"{BASE_URL}/step", json={"action": action_dict})
    data = res.json()
    return data['observation'], data['reward'], data['done']

print("Environment wrappers initialized!")

Environment wrappers initialized!


In [2]:
# Example Dummy Output from a Policy Network (PyTorch)
obs = wrap_env_reset('task_3')
done = False
total_reward = 0.0

print(f"Starting trial. Initial Budget: {obs['budget_remaining']}")

while not done:
    # Provide a heuristic wrapper action (Here you would instead do `policy_net(obs)`)
    action = {
        "n_next_cohort": 30,
        "allocation_control": 0.25,
        "allocation_low": 0.25,
        "allocation_mid": 0.25,
        "allocation_high": 0.25,
        "stop_for_success": False,
        "stop_for_futility": False,
        "drop_arm": None,
        "inclusion_criteria_strictness": 0.8  # Aggressive strictness for max effect size
    }
    
    obs, reward, done = wrap_env_step(action)
    total_reward += reward
    
    print(f"[Interim {obs['interim_number']}] Budget Left: {obs['budget_remaining']} | "
          f"P-values [L: {obs['p_value_low']}, M: {obs['p_value_mid']}, H: {obs['p_value_high']}]")

print(f"\nTrial Ended. Total Grader Reward: {total_reward}")

Starting trial. Initial Budget: 150
[Interim 1] Budget Left: 65 | P-values [L: 0.4667, M: 1.0, H: 0.6084]
[Interim 1] Budget Left: 65 | P-values [L: 0.2821, M: 1.0, H: 0.5692]
[Interim 1] Budget Left: 65 | P-values [L: 1.0, M: 0.0256, H: 0.0014]
[Interim 1] Budget Left: 65 | P-values [L: 1.0, M: 1.0, H: 0.1189]

Trial Ended. Total Grader Reward: 0.251476
